# Merge Amharic ASR Datasets → One Hugging Face Dataset

This notebook combines four Amharic speech datasets into a single unified ASR corpus and pushes it to your Hugging Face Hub account.

Source datasets:
- `badrex/amharic-speech`
- `chappM/amharic-bdu-asr`
- `beimnet777/amharic-asr`
- `snapwre/amharic-speech`

**Workflow:**
1. Install deps & log in to Hugging Face
2. Load each dataset and **inspect its raw schema** (column names differ across datasets, so don't skip this)
3. Standardize each one to a common schema: `audio`, `text`, `source`
4. Concatenate all four
5. Normalize Amharic text, resample audio, drop empty/broken rows
6. Deduplicate (by transcript text, and optionally by audio hash)
7. Re-split into train/validation/test
8. Push the merged dataset to the Hub

> ⚠️ Before pushing publicly, check each source dataset's license/card to confirm redistribution as a merged dataset is allowed. Most CC-BY style licenses are fine, but always verify.

## 1. Setup

In [ ]:
!pip install -q -U datasets huggingface_hub soundfile librosa

In [ ]:
from huggingface_hub import notebook_login

# Paste a HF token with WRITE access when prompted (huggingface.co/settings/tokens)
notebook_login()

In [ ]:
from datasets import load_dataset, concatenate_datasets, Audio, DatasetDict, Dataset
import re
import unicodedata

SOURCE_REPOS = [
    "badrex/amharic-speech",
    "chappM/amharic-bdu-asr",
    "beimnet777/amharic-asr",
    "snapwre/amharic-speech",
]

## 2. Inspect each dataset's schema

Run this first and actually read the output — column names (e.g. `text` vs `transcript` vs `sentence`) and split names vary across these repos. You'll use what you see here to fill in the `COLUMN_MAP` in the next section.

In [ ]:
from datasets import get_dataset_config_names, get_dataset_split_names

for repo in SOURCE_REPOS:
    print(f"\n{'='*60}\n{repo}\n{'='*60}")
    try:
        configs = get_dataset_config_names(repo)
        print("configs:", configs)
        for cfg in configs:
            splits = get_dataset_split_names(repo, cfg)
            print(f"  config={cfg} splits={splits}")
    except Exception as e:
        print("Could not list configs/splits:", e)

    try:
        # Load just the first split lazily and peek at one example
        splits = get_dataset_split_names(repo)
        first_split = splits[0]
        preview = load_dataset(repo, split=f"{first_split}[:1]")
        print("features:", preview.features)
        print("example keys:", list(preview[0].keys()))
    except Exception as e:
        print("Could not preview:", e)

## 3. Configure column mapping

Update `COLUMN_MAP` below based on what the inspection cell printed for each repo. Each entry maps `repo_id -> {"text": <transcript column name>, "audio": <audio column name>, "splits": [<split names to include>]}`.

The defaults below are my best guess from the dataset viewers as of writing — **verify against your own inspection output above before running the merge**, since these can change.

In [ ]:
import requests

def check_schema(repo_id):
    url = f"https://datasets-server.huggingface.co/info?dataset={repo_id}"
    r = requests.get(url).json()
    if "error" in r:
        print(f"{repo_id}: ERROR -> {r['error']}")
        return
    for config, info in r["dataset_info"].items():
        print(f"\n{repo_id} [{config}]")
        for feat_name, feat_type in info["features"].items():
            print(f"  {feat_name}: {feat_type.get('_type') or feat_type.get('dtype')}")
        print(f"  splits: {list(info['splits'].keys())}")

for repo in [
    "badrex/amharic-speech",
    "chappM/amharic-bdu-asr",
    "beimnet777/amharic-asr",
    "snapwre/amharic-speech",
]:
    check_schema(repo)

In [ ]:
COLUMN_MAP = {
    "badrex/amharic-speech": {
        "text": "transcription",   # not "text"
        "audio": "audio",
        "splits": None,            # has validation/test/train — use all
    },
    "chappM/amharic-bdu-asr": {
        "text": "sentence",        # not "text"
        "audio": "audio",
        "splits": ["train", "test"],
    },
    "beimnet777/amharic-asr": {
        "text": "text",            # correct as-is
        "audio": "audio",
        "splits": ["train", "test"],
    },
    "snapwre/amharic-speech": {
        "text": "sentence",        # not "text"
        "audio": "audio",
        "splits": None,            # has train/validation/test — use all
    },
}

## 4. Load + standardize each dataset

Every dataset gets reduced to three columns: `audio`, `text`, `source`.

In [ ]:
def load_and_standardize(repo_id, cfg):
    text_col = cfg["text"]
    audio_col = cfg["audio"]
    splits = cfg["splits"]

    if splits is None:
        splits = get_dataset_split_names(repo_id)

    per_split = []
    for split in splits:
        ds = load_dataset(repo_id, split=split)

        rename_map = {}
        if text_col != "text":
            rename_map[text_col] = "text"
        if audio_col != "audio":
            rename_map[audio_col] = "audio"
        if rename_map:
            ds = ds.rename_columns(rename_map)

        # Keep only the columns we need
        cols_to_drop = [c for c in ds.column_names if c not in ("text", "audio")]
        if cols_to_drop:
            ds = ds.remove_columns(cols_to_drop)

        ds = ds.cast_column("audio", Audio(sampling_rate=16000))
        ds = ds.add_column("source", [f"{repo_id}:{split}"] * len(ds))
        per_split.append(ds)

    return concatenate_datasets(per_split)


standardized_datasets = []
for repo in SOURCE_REPOS:
    print(f"Loading {repo} ...")
    ds = load_and_standardize(repo, COLUMN_MAP[repo])
    print(f"  -> {len(ds)} rows")
    standardized_datasets.append(ds)

## 5. Concatenate into one dataset

In [ ]:
merged = concatenate_datasets(standardized_datasets)
print(f"Total merged rows (before cleaning): {len(merged)}")
merged

## 6. Clean & normalize text

- Strip surrounding whitespace and collapse internal whitespace
- Drop empty transcripts
- Unicode-normalize (NFC) so visually-identical Ge'ez characters are byte-identical across sources

In [ ]:
def normalize_text(example):
    text = example["text"] or ""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\s+", " ", text).strip()
    example["text"] = text
    return example

merged = merged.map(normalize_text)
before = len(merged)
merged = merged.filter(lambda ex: len(ex["text"]) > 0)
print(f"Dropped {before - len(merged)} rows with empty transcripts")
print(f"Remaining rows: {len(merged)}")

## 7. Deduplicate

Removes exact-duplicate transcripts (common when the same source recordings appear in more than one of the four datasets). Keeps the first occurrence.

In [ ]:
seen = set()
keep_indices = []
for i, text in enumerate(merged["text"]):
    key = text.strip().lower()
    if key not in seen:
        seen.add(key)
        keep_indices.append(i)

print(f"Dropping {len(merged) - len(keep_indices)} duplicate-transcript rows")
merged = merged.select(keep_indices)
print(f"Remaining rows: {len(merged)}")

## 8. Re-split into train / validation / test

We ignore each source dataset's original split and re-split the merged pool, so there's no leakage between train and test caused by the same speaker/recording appearing in different source datasets' splits.

In [ ]:
merged = merged.shuffle(seed=42)

n = len(merged)
train_end = int(n * 0.90)
val_end = int(n * 0.95)

dataset_dict = DatasetDict({
    "train": merged.select(range(0, train_end)),
    "validation": merged.select(range(train_end, val_end)),
    "test": merged.select(range(val_end, n)),
})

dataset_dict

## 9. Sanity check

Listen to / inspect a couple of rows before pushing, to make sure the audio and text actually line up correctly.

In [ ]:
import IPython.display as ipd

for i in range(3):
    ex = dataset_dict["train"][i]
    print(f"[{ex['source']}] {ex['text']}")
    display(ipd.Audio(ex["audio"]["array"], rate=ex["audio"]["sampling_rate"]))

In [ ]:
import os

def add_duration(example):
    example["duration_sec"] = len(example["audio"]["array"]) / example["audio"]["sampling_rate"]
    return example

num_proc = min(4, os.cpu_count() or 1)

total_hours_all = 0.0
for split_name, split_ds in dataset_dict.items():
    split_ds = split_ds.map(add_duration, num_proc=num_proc)
    dataset_dict[split_name] = split_ds  # keep the duration_sec column for reuse

    total_seconds = sum(split_ds["duration_sec"])
    hours = total_seconds / 3600
    total_hours_all += hours
    print(f"{split_name:>12}: {hours:7.2f} hours   ({len(split_ds):>7} clips,  avg {total_seconds/len(split_ds):.2f}s/clip)")

print(f"{'TOTAL':>12}: {total_hours_all:7.2f} hours   ({sum(len(d) for d in dataset_dict.values()):>7} clips)")

In [ ]:
def compute_total_hours(ds, batch_size=500):
    total_seconds = 0.0
    n = len(ds)
    for i in range(0, n, batch_size):
        batch = ds[i:i + batch_size]  # decodes audio for this batch only, in memory
        for audio in batch["audio"]:
            total_seconds += len(audio["array"]) / audio["sampling_rate"]
    return total_seconds / 3600

total_hours_all = 0.0
for split_name, split_ds in dataset_dict.items():
    hours = compute_total_hours(split_ds)
    total_hours_all += hours
    print(f"{split_name:>12}: {hours:7.2f} hours   ({len(split_ds):>7} clips,  avg {hours*3600/len(split_ds):.2f}s/clip)")

print(f"{'TOTAL':>12}: {total_hours_all:7.2f} hours   ({sum(len(d) for d in dataset_dict.values()):>7} clips)")

In [ ]:
import random

def estimate_hours(ds, sample_size=500, seed=42):
    n = len(ds)
    sample_size = min(sample_size, n)
    idx = random.Random(seed).sample(range(n), sample_size)
    sample = ds.select(idx)
    total_sec = sum(len(ex["audio"]["array"]) / ex["audio"]["sampling_rate"] for ex in sample)
    avg_sec = total_sec / sample_size
    return (avg_sec * n) / 3600

for split_name, split_ds in dataset_dict.items():
    est = estimate_hours(split_ds)
    print(f"{split_name:>12}: ~{est:.2f} hours (estimated from {min(500, len(split_ds))}-row sample)")

## 10. Push to the Hugging Face Hub

Set `HUB_REPO_ID` to `your-username/dataset-name`. Set `PRIVATE = True` first if you want to review it privately before making it public.

In [ ]:
HUB_REPO_ID = "Harbidel/amharic-asr-merged"  # <-- change this
PRIVATE = True  # flip to False when you're ready to make it public

dataset_dict.push_to_hub(HUB_REPO_ID, private=PRIVATE)
print(f"Pushed to https://huggingface.co/datasets/{HUB_REPO_ID}")

## 11. Write a dataset card

This creates a basic README.md on the Hub repo crediting the four source datasets.

In [ ]:
from huggingface_hub import HfApi

readme = f"""---
language:
- am
license: cc-by-4.0
task_categories:
- automatic-speech-recognition
pretty_name: Merged Amharic ASR Dataset
---

# {HUB_REPO_ID.split('/')[-1]}

A merged Amharic speech-recognition dataset, combining and deduplicating:

- [badrex/amharic-speech](https://huggingface.co/datasets/badrex/amharic-speech)
- [chappM/amharic-bdu-asr](https://huggingface.co/datasets/chappM/amharic-bdu-asr)
- [beimnet777/amharic-asr](https://huggingface.co/datasets/beimnet777/amharic-asr)
- [snapwre/amharic-speech](https://huggingface.co/datasets/snapwre/amharic-speech)

## Processing

- Standardized to `audio` (16kHz mono) and `text` columns, with a `source` column tracking origin
- Unicode NFC-normalized transcripts, empty transcripts dropped
- Exact-duplicate transcripts removed
- Re-split into train (90%) / validation (5%) / test (5%), ignoring original source splits

## Rows & duration

- Train: 67,221 clips (218.98 hours)
- Validation: 3,735 clips (12.25 hours)
- Test: 3,735 clips (12.19 hours)
- **Total: 74,691 clips (243.42 hours)**

## License

Check the license of each source dataset before redistributing; this card assumes CC-BY-4.0 as the most permissive common denominator among the sources at time of writing — verify this still holds for all four before relying on it.
"""

api = HfApi()
api.upload_file(
    path_or_fileobj=readme.encode(),
    path_in_repo="README.md",
    repo_id=HUB_REPO_ID,
    repo_type="dataset",
)
print("README.md uploaded.")
print(f"View it at: https://huggingface.co/datasets/{HUB_REPO_ID}")